# m50 lab · Intracardiac echocardiography: ultrasound essentials

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You turn echo times into depths and see what a wrong speed of sound does; find when two echoes stop overlapping;
budget the signal lost to attenuation; steer a phased array and make grating lobes appear; form a sector image of
a synthetic atrial septum with delay-and-sum beamforming and measure how far a sheath tip tents it; build a 3-D
shell from tracked imaging planes; and read the published ICE studies by their design.

| Part | What you do | Data |
| --- | --- | --- |
| A | Echo time to depth, and the speed-of-sound assumption (eq-m50-range) | published values |
| B | When two echoes stop overlapping (eq-m50-axial) | our own pulses |
| C | Attenuation and how deep an image reaches (eq-m50-attenuation) | published attenuation slopes |
| D | Steering, main lobe and grating lobes (eq-m50-steer) | our own array |
| E | A sector image of the septum, tenting, and frame rate (eq-m50-frame-rate) | our own phantom |
| F | A 3-D shell from tracked 2-D planes | our own phantom |
| G | Reading the ICE evidence by design | published results |

Defaults come from published sources where one exists: the 1540 m/s that scanners assume (Zhang et al. 2026),
ICE imaging frequencies of 7.5–10 MHz (Morton et al. 2003), myocardial attenuation slopes of 0.52 and
1.88 dB/(cm·MHz) (Baldwin et al. 2007). Pulse length, element count, signal budget, depths and phantom shapes are
illustrative. Each part ends with **checks**; the lab runs in under a minute and needs no downloads.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import ice

## Part A · Echo time to depth, and the speed-of-sound assumption

A scanner turns the time $t$ between sending a pulse and hearing its echo into depth with $z = ct/2$
(eq-m50-range), using one assumed speed of sound, 1540 m/s (Zhang et al. 2026). Real tissue differs: for example,
calf muscle measured 1560–1586 m/s in vivo (Xiao et al. 2024), used here only to show the size of the effect. A
reflector at true depth $z$ is then drawn at $z\,c_{\text{assumed}}/c_{\text{true}}$. Take the mean
cavotricuspid isthmus length measured with ICE, 35 mm (Morton et al. 2003).

In [ ]:
z_true = 35.0
for c_true in (1540.0, 1560.0, 1586.0):
    shown = ice.displayed_depth(z_true, c_true)
    print(f"tissue at {c_true:.0f} m/s: drawn at {shown:.2f} mm (error {shown - z_true:+.2f} mm)")

# the same through a simulated echo: send, receive, convert with 1540 m/s
d_axis, s = ice.a_line([z_true], [1.0], 7.5, 2, c=1586.0, max_depth_mm=45.0)
t_echo = 2 * d_axis[np.argmax(ice.envelope(s))] * 1e-3 / 1586.0
shown_sim = ice.depth_from_time(t_echo) * 1e3
z = np.linspace(0, 120, 121)
plt.plot(z, ice.displayed_depth(z, 1586.0) - z, label="tissue at 1586 m/s")
plt.plot(z, ice.displayed_depth(z, 1560.0) - z, label="tissue at 1560 m/s")
plt.xlabel("true depth (mm)"); plt.ylabel("depth error (mm)"); plt.legend(); plt.show()

In [ ]:
# Checks
assert abs(shown_sim - ice.displayed_depth(z_true, 1586.0)) < 0.02
assert ice.displayed_depth(z_true, 1540.0) == z_true

The error grows in proportion to depth: a few per cent of every distance, not a fixed offset. A tracked ICE
contour placed in 3-D inherits it.

## Part B · When two echoes stop overlapping

A pulse of $n$ cycles is $n\lambda$ long, with $\lambda = c/f$. Two reflectors produce separate echoes only when
their round-trip delays differ by more than the pulse duration, that is when they are more than $n\lambda/2$ apart
(eq-m50-axial). With 2-cycle pulses (illustrative) at the reported ICE frequencies of 7.5 and 10 MHz (Morton et al.
2003):

In [ ]:
rows = []
for f in ice.ICE_FREQ_MHZ:
    r = ice.axial_resolution_mm(f, 2)
    near, far = ice.echoes_overlap(20.0, 20.0 + 0.8 * r, f, 2), ice.echoes_overlap(20.0, 20.0 + 1.2 * r, f, 2)
    rows.append((f, r, near, far))
    print(f"{f:4.1f} MHz: wavelength {ice.wavelength_mm(f):.3f} mm, axial resolution {r:.3f} mm; "
          f"echoes overlap at 0.8x: {near}, at 1.2x: {far}")

f, r = 7.5, ice.axial_resolution_mm(7.5, 2)
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, k in zip(axes, (0.6, 1.4)):
    d, s = ice.a_line([20.0, 20.0 + k * r], [1.0, 1.0], f, 2, max_depth_mm=21.0)
    m = (d > 19.6) & (d < 20.9)
    ax.plot(d[m], s[m], lw=0.8, label="echo signal"); ax.plot(d[m], ice.envelope(s)[m], label="envelope")
    ax.set_title(f"reflectors {k * r:.2f} mm apart"); ax.set_xlabel("depth (mm)")
axes[0].legend(); plt.show()
print("For scale: ICE measured isthmus walls 2.7-4.1 mm thick before ablation (Morton et al. 2003).")

In [ ]:
# Checks
for f, r, near, far in rows:
    assert near and not far
assert ice.axial_resolution_mm(10.0, 2) < ice.axial_resolution_mm(7.5, 2)

## Part C · Attenuation and how deep an image reaches

Attenuation in soft tissue rises roughly in proportion to frequency, so a round trip to depth $z$ costs
$2\alpha f z$ decibels (eq-m50-attenuation). Freshly excised myocardium measured 0.52 dB/(cm·MHz) across the fibers and
1.88 dB/(cm·MHz) along them (Baldwin et al. 2007). We use these for the whole path, a stated simplification (the
path in ICE is mostly blood, which is not modeled separately), and an illustrative 60 dB signal budget.

In [ ]:
budget = 60.0
reach = {}
for name, a in ice.MYOCARDIUM_DB_PER_CM_MHZ.items():
    for f in ice.ICE_FREQ_MHZ:
        reach[(name, f)] = ice.max_depth_cm(budget, a, f)
        print(f"{name:13s} {f:4.1f} MHz: budget used up at {reach[(name, f)]:.1f} cm")
zc = np.linspace(0, 12, 121)
for f in ice.ICE_FREQ_MHZ:
    plt.plot(zc, ice.round_trip_loss_db(0.52, f, zc), label=f"{f} MHz, across fibers")
plt.axhline(budget, ls="--", color="gray", label="budget (illustrative)")
plt.xlabel("depth (cm)"); plt.ylabel("round-trip loss (dB)"); plt.legend(); plt.show()

In [ ]:
# Checks
assert reach[("across fibers", 7.5)] > reach[("across fibers", 10.0)]
assert abs(reach[("across fibers", 7.5)] / reach[("along fibers", 7.5)] - 1.88 / 0.52) < 1e-9

## Part D · Steering, main lobe and grating lobes

Firing element $k$ at $x_k$ with delay $\tau_k = x_k \sin\theta_0 / c$ points the beam at angle $\theta_0$
(eq-m50-steer). The array factor shows how strongly the array responds in each direction. With element spacing
$d$, extra full-strength lobes appear where $\sin\theta = \sin\theta_0 + m\lambda/d$; none can appear for
any steering angle when $d \le \lambda/2$. The main lobe narrows as the aperture $n d$ grows. The element counts
here are illustrative.

In [ ]:
theta = np.radians(np.linspace(-90, 90, 3601))
steer = np.radians(30.0)
for d_over_lambda in (0.5, 1.0):
    af = ice.array_factor(theta, 32, d_over_lambda, steer)
    plt.plot(np.degrees(theta), 20 * np.log10(np.maximum(af, 1e-3)), label=f"spacing {d_over_lambda} wavelengths")
    print(f"spacing {d_over_lambda}: grating lobes at {ice.grating_lobes_deg(d_over_lambda, steer)} degrees")
plt.ylim(-50, 3); plt.xlabel("angle (degrees)"); plt.ylabel("array factor (dB)"); plt.legend(); plt.show()
widths = {n: np.degrees(np.arcsin(ice.first_null_offset(n, 0.5))) for n in (16, 32, 64)}
print("first null from the unsteered main lobe:", {n: round(w, 2) for n, w in widths.items()}, "degrees")

In [ ]:
# Checks
assert ice.grating_lobes_deg(0.5, steer) == []
assert len(ice.grating_lobes_deg(1.0, steer)) == 1
assert widths[64] < widths[32] < widths[16]

## Part E · A sector image of the septum, tenting, and frame rate

A synthetic interatrial septum lies 25 mm in front of a 32-element array (illustrative). Its thinner central part,
the fossa ovalis, reflects less. A sheath tip pressing on the fossa pushes it away from the probe (tenting). We
simulate the echoes on each element and form the image by delay-and-sum: for each pixel, add every element's
signal at that pixel's travel time. Each image line must wait for echoes from the deepest point, so a frame of
$N$ lines to depth $D$ takes $2DN/c$ (eq-m50-frame-rate).

In [ ]:
lam = ice.wavelength_mm(7.5) * 1e-3
pitch, n = lam / 2, 32
angles = np.radians(np.linspace(-35, 35, 71))
radii = np.linspace(0.015, 0.035, 401)
images, measured = {}, {}
for tent in (0.0, 4.0):
    pts, amp = ice.septum_scatterers(tent_mm=tent)
    t, ch = ice.simulate_channels(pts, amp, n, pitch, 7.5, 2, max_depth_m=0.04)
    images[tent] = ice.das_sector(t, ch, n, pitch, angles, radii)
    measured[tent] = ice.peak_depth_on_axis(images[tent], radii, angles) - 25.0
    print(f"tent {tent:.0f} mm: measured {measured[tent]:.2f} mm on the central line")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
A, R = np.meshgrid(angles, radii, indexing="ij")
for ax, tent in zip(axes, images):
    img = images[tent] / images[tent].max()
    ax.pcolormesh(R * np.sin(A) * 1e3, R * np.cos(A) * 1e3, 20 * np.log10(np.maximum(img, 1e-3)),
                  vmin=-40, vmax=0, cmap="gray", shading="gouraud")
    ax.set_title(f"tent {tent:.0f} mm"); ax.set_xlabel("lateral (mm)")
axes[0].set_ylabel("depth (mm)"); axes[0].invert_yaxis(); plt.show()

for depth_cm in (4, 8, 12):
    print(f"depth {depth_cm} cm, 96 lines: {ice.frame_rate(depth_cm / 100, 96):.0f} frames per second")

In [ ]:
# Checks
assert abs(measured[0.0]) < 0.2 and abs(measured[4.0] - 4.0) < 0.2
assert abs(ice.frame_rate(0.04, 96) / ice.frame_rate(0.12, 96) - 3.0) < 1e-9

The fossa is dimmer than the rest of the septum because it reflects less in this phantom, and the tent is
measured to within a fraction of a millimeter because the simulation knows the speed of sound exactly (part A says
what happens when it does not).

## Part F · A 3-D shell from tracked 2-D planes

When the ICE catheter is tracked by the mapping system, each imaging plane has a known position, and contours drawn
on it become 3-D points. Here an ellipsoidal chamber (semi-axes 25, 20 and 30 mm, illustrative) is cut by planes
that rotate about the catheter axis, with 0.5 mm of contour noise (illustrative). Turning more often reduces the
largest unsampled gap between planes; the surface between them must be interpolated (m20).

In [ ]:
rng = np.random.default_rng(3)
summary = {}
for n_planes in (3, 6, 12):
    pts = ice.ellipsoid_contours(n_planes, noise_mm=0.5, rng=rng)
    rms = float(np.sqrt(np.mean(ice.distance_to_ellipsoid(pts) ** 2)))
    summary[n_planes] = (len(pts), rms, ice.largest_gap_mm(n_planes))
    print(f"{n_planes:2d} planes: {len(pts):4d} points, RMS distance to the true surface {rms:.2f} mm, "
          f"largest gap between planes {ice.largest_gap_mm(n_planes):.1f} mm")
pts = ice.ellipsoid_contours(6, noise_mm=0.5, rng=rng)
plt.figure(figsize=(4, 4)); plt.scatter(pts[:, 0], pts[:, 1], s=3)
plt.gca().set_aspect("equal"); plt.xlabel("x (mm)"); plt.ylabel("y (mm)"); plt.title("6 planes, seen along the catheter"); plt.show()

In [ ]:
# Checks
assert summary[12][2] < summary[6][2] < summary[3][2]
assert all(0.2 < v[1] < 0.6 for v in summary.values())     # noise, not the number of planes, sets point error

## Part G · Reading the ICE evidence by design

The numbers below are as the cited studies report them. The point is the design: what was compared, how, and how
certain the authors say the result is.

In [ ]:
evidence = [
    ("Abdalla 2026", "ICE vs TEE/fluoroscopy for transseptal puncture", "meta-analysis, randomized and observational, 8 studies",
     "fluoroscopy time -2.07 min; no increase in safety events"),
    ("Kanniyan 2026", "ICE vs TEE for AF ablation", "meta-analysis, 5 cohorts and 1 randomized trial, 4,760 patients",
     "no significant difference in complications; fluoroscopy -4.17 min; certainty low to very low"),
    ("Yaser 2026", "ICE vs TEE for appendage occlusion", "meta-analysis, 27 observational studies",
     "similar success; more effusions needing intervention with ICE (adjusted OR 1.74)"),
    ("Deshpande 2022", "ICE vs no ICE in AF ablation", "administrative database, propensity matched, 51,129 patients",
     "90-day readmission hazard ratio 0.88"),
    ("Kitamura 2021", "ICE vs no ICE in VT ablation", "administrative database, 1,147 matched pairs",
     "less cardiac tamponade; no other difference"),
    ("Dello Russo 2023", "ICE vs no ICE in pulsed-field ablation", "multicenter cohort, 138 matched pairs",
     "no improvement in procedural metrics"),
    ("Chen 2026", "two ICE-guided transseptal strategies", "randomized trial, 43 patients",
     "zero fluoroscopy in all; no complications"),
]
for who, question, design, result in evidence:
    print(f"{who:16s} | {question:46s} | {design:62s} | {result}")
randomized_ice_vs_tee = [e for e in evidence if "TEE" in e[1] and e[2].startswith("randomized")]
print("Studies in this list that are themselves a randomized ICE-versus-TEE trial:", len(randomized_ice_vs_tee))

In [ ]:
# Checks
assert len(evidence) == 7 and len(randomized_ice_vs_tee) == 0

Most of the evidence on ICE is observational or pooled from observational studies; the one randomized trial in
this list compares two ways of using ICE, not ICE against something else. "No significant difference" in an
imprecise meta-analysis is not proof of equivalence (Kanniyan et al. 2026).

## Where to go next

- In part A, put the septum at 25 mm in tissue at 1586 m/s and image it with part E's code. Where is it drawn?
- In part B, use 4-cycle pulses. Which isthmus wall thickness from Morton et al. could you still resolve?
- In part D, add a random delay error to each element. What happens to the side lobes?
- In part E, make the fossa reflect as strongly as the rest of the septum. Can you still find it?